In [1]:
import os, sys, glob, time, warnings, collections

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np 
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.decomposition   import PCA
from sklearn.metrics         import (accuracy_score, f1_score,
                                     classification_report, confusion_matrix)
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm             import SVC
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.neural_network  import MLPClassifier

In [4]:
import torch, torch.nn as nn 
from torch.utils.data import DataLoader, TensorDataset

In [5]:
def load_one_file(path, short_threshold=1e-5):
    """
    Load one QFlow lite .npy file.
 
    Parameters
    ----------
    path             : str   path to the .npy file
    short_threshold  : float mean |current| (A) threshold to call Short Circuit
                             (only used when max pixel state == 0)
 
    Returns
    -------
    current  : (H, W) float32
    sensor0  : (H, W) float32
    sensor1  : (H, W) float32
    state_map: (H, W) int8
    label    : int   0=Barrier  1=Single Dot  2=Double Dot  3=Short Circuit
    """
    data   = np.load(path, allow_pickle=True).item()
    output = data["output"]          # array of dicts, one per pixel
 
    n_px = len(output)
    side = int(round(n_px ** 0.5))  # infer grid side from data (100 for QFlow lite)
 
    current   = np.array([o["current"]   for o in output], dtype=np.float32)
    sensor0   = np.array([o["sensor"][0] for o in output], dtype=np.float32)
    sensor1   = np.array([o["sensor"][1] for o in output], dtype=np.float32)
    state_arr = np.array([o["state"]     for o in output], dtype=np.int8)
 
    current   = current.reshape(side, side)
    sensor0   = sensor0.reshape(side, side)
    sensor1   = sensor1.reshape(side, side)
    state_map = state_arr.reshape(side, side)
 
    max_state = int(state_arr.max())
 
    if max_state == 1:
        label = 1
    elif max_state == 2:
        label = 2
    else:                            # max_state == 0: barrier or short circuit
        mean_cur = float(np.abs(current).mean())
        label = 3 if mean_cur > short_threshold else 0
 
    return current, sensor0, sensor1, state_map, label

In [6]:
def load_data(data_dir, short_threshold=1e-5,
                 class_names=None):
    """
    Load every .npy file in data_dir.
 
    Parameters
    ----------
    data_dir         : str    folder containing .npy files
    short_threshold  : float  passed to load_one_file
    class_names      : list   used only for the printed summary
 
    Returns
    -------
    X_cur  : (N, H*W) float32
    X_sen0 : (N, H*W) float32
    X_sen1 : (N, H*W) float32
    X_smap : (N, H*W) float32
    y      : (N,)     int64
    paths  : list[str]
    """
    if class_names is None:
        class_names = ["Barrier", "Single Dot", "Double Dot", "Short Circuit"]
 
    npy_files = sorted(glob.glob(os.path.join(data_dir, "*.npy")))
    if not npy_files:
        print(f"  [!] No .npy files found in '{data_dir}'.")
        return None
 
    print(f"  Found {len(npy_files)} .npy files – loading ...")
    t0 = time.time()
 
    currents, sens0, sens1, smaps, labels = [], [], [], [], []
    failed = 0
 
    for i, path in enumerate(npy_files):
        try:
            cur, s0, s1, sm, lbl = load_one_file(path, short_threshold)
            currents.append(cur.ravel())
            sens0.append(s0.ravel())
            sens1.append(s1.ravel())
            smaps.append(sm.ravel())
            labels.append(lbl)
        except Exception as e:
            failed += 1
            print(f"    skip {os.path.basename(path)}: {e}")
 
        if (i + 1) % 200 == 0:
            print(f"    {i + 1}/{len(npy_files)} ...")
 
    N = len(labels)
    print(f"  Loaded {N} samples in {time.time() - t0:.1f}s  ({failed} failed)")
    dist = collections.Counter(labels)
    print(f"  Class distribution: { {class_names[k]: v for k, v in sorted(dist.items())} }")
 
    return (
        np.array(currents, dtype=np.float32),
        np.array(sens0,    dtype=np.float32),
        np.array(sens1,    dtype=np.float32),
        np.array(smaps,    dtype=np.float32),
        np.array(labels,   dtype=np.int64),
        npy_files[:N],
    )

In [9]:
dat = load_data("C:/Users/Lehma/AIProjects/data_qflow_lite")

  Found 1001 .npy files – loading ...
    200/1001 ...
    400/1001 ...
    600/1001 ...
    800/1001 ...
    1000/1001 ...
  Loaded 1001 samples in 173.0s  (0 failed)
  Class distribution: {'Double Dot': 1001}


In [14]:
CFG = {
    "data_dir"       : "data",
    "results_dir"    : "results",
    "seed"           : 42,
    "img_size"       : 100,
    "short_threshold": 1e-5,
    "pca_components" : 100,
    "cv_folds"       : 5,
    "test_size"      : 0.20,
    "class_names"    : ["Barrier", "Single Dot", "Double Dot", "Short Circuit"],
    "class_colors"   : ["#4C72B0", "#55A868", "#C44E52", "#8172B2"],
}
 
np.random.seed(CFG["seed"])
os.makedirs(CFG["results_dir"], exist_ok=True)

In [10]:
def plot_samples(X_cur, X_s0, X_s1, X_sm, y,
                 class_names, results_dir, img_size=100, n_per_class=3):
    n_cols = n_per_class * 3
    n_rows = len(class_names)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_per_class * 7, 11),
                             constrained_layout=True)
    fig.suptitle("QFlow Lite – Sample Maps per Device State", fontsize=14)
 
    cmaps      = ["viridis", "plasma", "tab10"]
    col_titles = ["Current", "Sensor 0", "State Map"]
 
    for cls_idx, cls_name in enumerate(class_names):
        idxs = np.where(y == cls_idx)[0][:n_per_class]
        for j, si in enumerate(idxs):
            maps = [
                X_cur[si].reshape(img_size, img_size),
                X_s0[si].reshape(img_size, img_size),
                X_sm[si].reshape(img_size, img_size),
            ]
            for k, (mdata, cmap) in enumerate(zip(maps, cmaps)):
                ax = axes[cls_idx, j * 3 + k]
                ax.imshow(mdata, cmap=cmap, origin="lower", aspect="auto")
                ax.set_xticks([]); ax.set_yticks([])
                if j == 0 and k == 0:
                    ax.set_ylabel(cls_name, fontsize=11, fontweight="bold")
                if cls_idx == 0 and j == 0:
                    ax.set_title(col_titles[k], fontsize=9)
 
    path = os.path.join(results_dir, "sample_maps.png")
    plt.savefig(path, dpi=120, bbox_inches="tight"); plt.close()
    print(f"  saved: {path}")

In [11]:
#Feature Matrix
def build_feature_matrix(X_cur, X_s0, X_s1, X_sm):
    """Concatenate all four flat maps -> (N, 4 * H * W)."""
    return np.concatenate([X_cur, X_s0, X_s1, X_sm], axis=1)

In [12]:
#Build a pipeline with the classical ML Models
def build_pipelines(n_pca=100, seed=42):
    pca_step = [("pca", PCA(n_components=n_pca, random_state=seed))]
    return {
        "Random Forest": Pipeline([
            ("scaler", StandardScaler()), *pca_step,
            ("clf", RandomForestClassifier(
                n_estimators=500, max_features="sqrt",
                n_jobs=-1, random_state=seed)),
        ]),
        "SVM (RBF)": Pipeline([
            ("scaler", StandardScaler()), *pca_step,
            ("clf", SVC(kernel="rbf", C=10, gamma="scale",
                        class_weight="balanced", random_state=seed)),
        ]),
        "Gradient Boosting": Pipeline([
            ("scaler", StandardScaler()), *pca_step,
            ("clf", GradientBoostingClassifier(
                n_estimators=300, max_depth=4,
                learning_rate=0.05, subsample=0.8,
                random_state=seed)),
        ]),
        "k-NN (k=7)": Pipeline([
            ("scaler", StandardScaler()), *pca_step,
            ("clf", KNeighborsClassifier(n_neighbors=7, n_jobs=-1)),
        ]),
        "MLP": Pipeline([
            ("scaler", StandardScaler()), *pca_step,
            ("clf", MLPClassifier(
                hidden_layer_sizes=(512, 256, 128, 64),
                activation="relu", solver="adam",
                batch_size=32, max_iter=500,
                early_stopping=True, validation_fraction=0.1,
                random_state=seed)),
        ]),
    }

In [13]:
#Evaluate Models with Cross-Validations
def evaluate_models(X, y, pipes, cv_folds=5, test_size=0.20, seed=42):
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=seed)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed)
 
    results = []
    for name, pipe in pipes.items():
        print(f"\n  +-- {name}")
        t0 = time.time()
        cv_scores = cross_val_score(pipe, X_tr, y_tr,
                                    cv=skf, scoring="accuracy", n_jobs=-1)
        print(f"  |   CV  Acc : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}"
              f"  ({time.time() - t0:.1f}s)")
        pipe.fit(X_tr, y_tr)
        y_pred   = pipe.predict(X_te)
        test_acc = accuracy_score(y_te, y_pred)
        test_f1  = f1_score(y_te, y_pred, average="weighted")
        print(f"  |   Test Acc: {test_acc:.4f}   Weighted F1: {test_f1:.4f}")
 
        results.append({
            "model"   : name,
            "cv_mean" : cv_scores.mean(),
            "cv_std"  : cv_scores.std(),
            "test_acc": test_acc,
            "test_f1" : test_f1,
            "y_true"  : y_te,
            "y_pred"  : y_pred,
        })
    return results, y_te

In [15]:
def train_cnn(X_cur, X_s0, X_s1, X_sm, y,
              class_names, results_dir,
              img_size=100, test_size=0.20,
              epochs=30, batch_size=32, lr=1e-3, seed=42):
 
    n_classes = len(class_names)
    device = torch.device("cpu")
 
    X_img = np.stack([
        X_cur.reshape(-1, img_size, img_size),
        X_s0.reshape(-1,  img_size, img_size),
        X_s1.reshape(-1,  img_size, img_size),
        X_sm.reshape(-1,  img_size, img_size),
    ], axis=1).astype(np.float32)
 
    for c in range(4):
        mu = X_img[:, c].mean(); sig = X_img[:, c].std() + 1e-8
        X_img[:, c] = (X_img[:, c] - mu) / sig
 
    n    = len(y)
    idx  = np.random.permutation(n)
    n_te = int(n * test_size); n_va = int(n * 0.10)
    tr_idx = idx[n_te + n_va:]; va_idx = idx[n_te:n_te + n_va]; te_idx = idx[:n_te]
 
    def make_loader(idxs, shuffle=True):
        return DataLoader(
            TensorDataset(torch.tensor(X_img[idxs]),
                          torch.tensor(y[idxs], dtype=torch.long)),
            batch_size=batch_size, shuffle=shuffle)
 
    tr_ldr = make_loader(tr_idx); va_ldr = make_loader(va_idx, False)
    te_ldr = make_loader(te_idx, False)
 
    class QDotCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.enc = nn.Sequential(
                nn.Conv2d(4,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
                nn.Conv2d(32,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
                nn.Conv2d(64,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
                nn.Conv2d(128,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
                nn.MaxPool2d(2), nn.AdaptiveAvgPool2d(4),
            )
            self.head = nn.Sequential(
                nn.Flatten(),
                nn.Linear(128*16, 256), nn.ReLU(), nn.Dropout(0.4),
                nn.Linear(256, 64),     nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(64, n_classes),
            )
        def forward(self, x): return self.head(self.enc(x))
 
    model     = QDotCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
 
    tr_accs, va_accs = [], []
    best_va = 0.0; best_state = None
 
    for epoch in range(1, epochs + 1):
        model.train(); c = t = 0
        for Xb, yb in tr_ldr:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(Xb); loss = criterion(out, yb)
            loss.backward(); optimizer.step()
            c += (out.argmax(1) == yb).sum().item(); t += len(yb)
        tr_accs.append(c / t)
 
        model.eval(); c = t = 0
        with torch.no_grad():
            for Xb, yb in va_ldr:
                Xb, yb = Xb.to(device), yb.to(device)
                c += (model(Xb).argmax(1) == yb).sum().item(); t += len(yb)
        va_acc = c / t; va_accs.append(va_acc); scheduler.step()
 
        if va_acc > best_va:
            best_va = va_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{epochs}  train={tr_accs[-1]:.4f}  val={va_acc:.4f}")
 
    model.load_state_dict(best_state); model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for Xb, yb in te_ldr:
            preds.extend(model(Xb.to(device)).argmax(1).cpu().numpy())
            trues.extend(yb.numpy())
 
    test_acc = accuracy_score(trues, preds)
    test_f1  = f1_score(trues, preds, average="weighted")
    print(f"\n  CNN Test Acc: {test_acc:.4f}   Weighted F1: {test_f1:.4f}")
    print(classification_report(trues, preds, target_names=class_names))
 
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(tr_accs, lw=2, label="Train")
    ax.plot(va_accs, lw=2, ls="--", label="Validation")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
    ax.set_title("CNN Learning Curves"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(results_dir, "cnn_learning_curves.png")
    plt.savefig(path, dpi=120, bbox_inches="tight"); plt.close()
    print(f"  saved: {path}")
 
    return {"model": "CNN", "cv_mean": None, "cv_std": None,
            "test_acc": test_acc, "test_f1": test_f1,
            "y_true": np.array(trues), "y_pred": np.array(preds)}

In [16]:
def plot_pca_2d(X, y, class_names, class_colors, results_dir, seed=42):
    Xs  = StandardScaler().fit_transform(X)
    pca = PCA(n_components=2, random_state=seed)
    Xp  = pca.fit_transform(Xs)
    fig, ax = plt.subplots(figsize=(7, 6))
    for i, (name, col) in enumerate(zip(class_names, class_colors)):
        m = y == i
        ax.scatter(Xp[m,0], Xp[m,1], c=col, label=name, alpha=0.55, s=25, edgecolors="none")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    ax.set_title("PCA 2D Projection"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [17]:
def plot_confusion_matrices(results, class_names, results_dir):
    n = len(results); ncols = 3; nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*5, nrows*4.5))
    axes = np.array(axes).ravel()
    for ax, r in zip(axes, results):
        cm = confusion_matrix(r["y_true"], r["y_pred"])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=[c[:3] for c in class_names],
                    yticklabels=[c[:3] for c in class_names],
                    ax=ax, cbar=False)
        ax.set_title(f"{r['model']}\nAcc={r['test_acc']:.3f}  F1={r['test_f1']:.3f}", fontsize=10)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    for ax in axes[n:]: ax.axis("off")
    fig.suptitle("Confusion Matrices – All Models", fontsize=14)
    plt.tight_layout()
    plt.show()

In [18]:
def plot_comparison(results, results_dir):
    names  = [r["model"]    for r in results]
    accs   = [r["test_acc"] for r in results]
    f1s    = [r["test_f1"]  for r in results]
    cv_mu  = [r["cv_mean"] if r["cv_mean"] is not None else r["test_acc"] for r in results]
    cv_std = [r["cv_std"]  if r["cv_std"]  is not None else 0.0           for r in results]
    x, w = np.arange(len(names)), 0.27
    fig, ax = plt.subplots(figsize=(13, 5))
    b1 = ax.bar(x-w, cv_mu, w, label="CV Accuracy",  color="#4C72B0", yerr=cv_std, capsize=4)
    b2 = ax.bar(x,   accs,  w, label="Test Accuracy", color="#55A868")
    b3 = ax.bar(x+w, f1s,   w, label="Weighted F1",   color="#C44E52")
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=12, ha="right")
    ax.set_ylim(0, 1.10); ax.set_ylabel("Score")
    ax.set_title("Model Comparison – QFlow Lite"); ax.legend(loc="lower right")
    ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)
    for bars in [b1, b2, b3]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2, h+0.005,
                    f"{h:.3f}", ha="center", va="bottom", fontsize=7)
    plt.tight_layout()
    plt.show()

In [19]:
def plot_rf_pixel_importance(X, y, results_dir, n_pca=100, img_size=100, seed=42):
    Xs  = StandardScaler().fit_transform(X)
    pca = PCA(n_components=n_pca, random_state=seed)
    Xp  = pca.fit_transform(Xs)
    rf  = RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=seed)
    rf.fit(Xp, y)
    imp_px = np.abs(pca.components_).T @ rf.feature_importances_
    px = img_size * img_size
    chan_names = ["Current", "Sensor 0", "Sensor 1", "State Map"]
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for i, (ax, cname) in enumerate(zip(axes, chan_names)):
        imap = imp_px[i*px:(i+1)*px].reshape(img_size, img_size)
        im = ax.imshow(imap, cmap="hot", origin="lower")
        ax.set_title(f"RF Importance\n{cname}", fontsize=10)
        ax.set_xlabel("V_P2 (px)"); ax.set_ylabel("V_P1 (px)")
        plt.colorbar(im, ax=ax, fraction=0.046)
    plt.suptitle("Random Forest – Feature Importance per Pixel Map", fontsize=12)
    plt.tight_layout()
    plt.show()

In [20]:
def print_summary(results, class_names):
    print("\n" + "="*74)
    print("  FINAL RESULTS")
    print("="*74)
    print(f"  {'Model':<25} {'CV Acc':>9} {'+-':>6} {'Test Acc':>10} {'F1(wt)':>8}")
    print("  " + "-"*63)
    for r in sorted(results, key=lambda x: x["test_acc"], reverse=True):
        cv  = f"{r['cv_mean']:.4f}" if r["cv_mean"] is not None else "   -  "
        std = f"{r['cv_std']:.4f}"  if r["cv_std"]  is not None else "   -  "
        print(f"  {r['model']:<25} {cv:>9} {std:>6} "
              f"{r['test_acc']:>10.4f} {r['test_f1']:>8.4f}")
    print("="*74)
    best = max(results, key=lambda r: r["test_acc"])
    print(f"\n  Best model: {best['model']}  (test acc={best['test_acc']:.4f})")
    print(classification_report(best["y_true"], best["y_pred"],
                                 target_names=class_names))